In [1]:
import json
import os
from typing import List, Tuple

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader, Sampler

from torch.amp.grad_scaler import GradScaler
from torch.amp.autocast_mode import autocast

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"

In [2]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [3]:
BASE_PATH = r"/kaggle/input/asl-signs"
TRAIN_FILE = r"/kaggle/input/asl-signs/train.csv"
SIGN_INDEX_FILE = r"/kaggle/input/asl-signs/sign_to_prediction_index_map.json"

with open(SIGN_INDEX_FILE, "r") as json_file:
    SIGN2INDEX_JSON = json.load(json_file)

INCLUDE_FACE = True  
INCLUDE_DEPTH = False

FACE_LANDMARK_INDICES = {
    'nose': [1, 2, 4, 5, 6, 19, 94, 168, 197, 195],
    'left_eye': [33, 133, 160, 159, 158, 157, 173, 144, 145, 153, 154, 155, 156, 246, 7, 163],
    'right_eye': [263, 362, 387, 386, 385, 384, 398, 373, 374, 380, 381, 382, 466, 388, 390, 249],
    'left_eyebrow': [70, 63, 105, 66, 107, 55, 65, 52],
    'right_eyebrow': [300, 293, 334, 296, 336, 285, 295, 282],
    'mouth_outer': [61, 146, 91, 181, 84, 17, 314, 405, 321, 375, 291, 409, 270, 269, 267, 0, 37, 39, 40, 185],
    'mouth_inner': [78, 191, 80, 81, 82, 13, 312, 311, 310, 415, 308, 324, 318, 402, 317, 14, 87, 178, 88, 95],
    'face_oval': [10, 338, 297, 332, 284, 251, 389, 356, 454, 323, 361, 288, 397, 365, 379, 378, 400, 377, 
                  152, 148, 176, 149, 150, 136, 172, 58, 132, 93, 234, 127, 162, 21, 54, 103, 67, 109]
}

SELECTED_FACE_INDICES = []
for feature_indices in FACE_LANDMARK_INDICES.values():
    SELECTED_FACE_INDICES.extend(feature_indices)

In [4]:
def generate_full_column_list() -> List[str]:
    """
    Generates the standardized list of 1629 column names (x/y/z for 543 landmarks).
    """
    landmark_specs = {
        "left_hand": 21,  # Indices 0 to 20
        "pose": 33,  # Indices 0 to 32
        "right_hand": 21,  # Indices 0 to 20
    }

    axes = ["x", "y", "z"] if INCLUDE_DEPTH else ["x", "y"]

    full_columns = []

    for landmark_type, count in landmark_specs.items():
        for i in range(count):
            for axis in axes:
                full_columns.append(f"{landmark_type}_{i}_{axis}")

    if INCLUDE_FACE:
        for face_idx in SELECTED_FACE_INDICES:
            for axis in axes:
                full_columns.append(f"face_{face_idx}_{axis}")

    return full_columns


ALL_COLUMNS = generate_full_column_list()

In [5]:
def normalize_values(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalize coordinates using the nose coordinates

    Args:
        df (pd.DataFrame): Unnormalaized dataframe

    Returns:
        pd.Dataframe: Normalized dataframe / series
    """
    axes = ["x", "y", "z"] if INCLUDE_DEPTH else ["x", "y"]

    origins = df[(df["type"] == "pose") & (df["landmark_index"] == 0)].set_index(
        "frame"
    )[axes]

    def normalize_frame(frame_df: pd.DataFrame) -> pd.DataFrame:
        frame = frame_df.name
        if frame not in origins.index:
            return frame_df  # or raise an error
        frame_df[axes] = frame_df[axes] - origins.loc[frame]
        return frame_df

    normalized_df = df.groupby("frame", group_keys=False).apply(normalize_frame)
    return normalized_df


def frame_stacked_data(file_path: str) -> np.ndarray:
    """
    Read landmark data from parquet files, normalize and stack them for each frame

    Args:
        file_path (str): File path for the parquet file

    Returns:
        np.ndarray: The normlaized stacked coordinates
    """
    df = pd.read_parquet(os.path.join(BASE_PATH, file_path))
    if INCLUDE_FACE:
        face_df = df[df["type"] == "face"]
        face_df = face_df[face_df["landmark_index"].isin(SELECTED_FACE_INDICES)]
        other_df = df[df["type"] != "face"]
        df = pd.concat([face_df, other_df], ignore_index=True)
    else:
        df = df[df["type"] != "face"]
    axes = ["x", "y", "z"] if INCLUDE_DEPTH else ["x", "y"]

    df["UID"] = df["type"].astype("str") + "_" + df["landmark_index"].astype("str")
    df = df.sort_values(["frame", "UID"])
    
    df = normalize_values(df)

    pivot_df = df.pivot_table(index="frame", columns="UID", values=axes)
    pivot_df.columns = [f"{col[1]}_{col[0]}" for col in pivot_df.columns]
    pivot_df = pivot_df.reindex(columns=ALL_COLUMNS)

    final_data = pd.DataFrame(pivot_df).ffill().bfill().fillna(0).to_numpy()
    return final_data

In [6]:
def augment_sample(
    video_coordinates: np.ndarray, noise_std: float = 3e-3, spatial_shift: float = 2e-2
) -> np.ndarray:
    video_coordinates = video_coordinates.copy()

    if np.random.random() > 0.5:
        noise = np.random.normal(0, noise_std, video_coordinates.shape)
        video_coordinates = video_coordinates + noise

    if np.random.random() > 0.5:
        shift = np.random.uniform(
            -spatial_shift, spatial_shift, (1, video_coordinates.shape[1])
        )
        video_coordinates = video_coordinates + shift

    if np.random.random() > 0.5 and video_coordinates.shape[0] > 20:
        start_idx = np.random.randint(0, max(1, video_coordinates.shape[0] // 10))
        end_idx = video_coordinates.shape[0] - np.random.randint(
            0, max(1, video_coordinates.shape[0] // 10)
        )
        video_coordinates = video_coordinates[start_idx:end_idx]

    return video_coordinates

In [7]:
class ASLDataset(Dataset):
    def __init__(self, video_coordinates, video_labels, max_frames=128, augment=False):
        self.video_coordinates = video_coordinates
        self.video_labels = video_labels
        self.max_frames = max_frames
        self.augment = augment

    def __len__(self):
        return len(self.video_coordinates)

    def __getitem__(self, idx):
        x = self.video_coordinates[idx]
        y = self.video_labels[idx]

        if self.augment:
            x = augment_sample(x)

        if x.shape[0] > self.max_frames:
            idxs = np.linspace(0, x.shape[0] - 1, self.max_frames).astype(int)
            x = x[idxs]

        vel = x[1:] - x[:-1]
        vel = np.vstack([np.zeros_like(x[:1]), vel])
        x = np.concatenate([x, vel], axis=1)

        return torch.tensor(x, dtype=torch.float32), y

In [8]:
def collate_fn(batch):
    sequences, labels = zip(*batch)

    lengths = torch.tensor([seq.shape[0] for seq in sequences])
    max_len = int(lengths.max())

    D = sequences[0].shape[1]
    B = len(sequences)

    padded = torch.zeros(B, max_len, D)
    mask = torch.zeros(B, max_len, dtype=torch.bool)

    for i, seq in enumerate(sequences):
        T = seq.shape[0]
        padded[i, :T] = seq
        mask[i, :T] = 1

    return padded, mask, torch.tensor(labels)

In [9]:
class BucketBatchSampler(Sampler):
    """
    Sampler that groups sequences by length into buckets and shuffles buckets each epoch
    """

    def __init__(self, lengths, batch_size, drop_last=False):
        self.lengths = lengths
        self.batch_size = batch_size
        self.drop_last = drop_last

    def __iter__(self):
        sorted_idxs = np.argsort(self.lengths)

        buckets = []
        for i in range(0, len(sorted_idxs), self.batch_size):
            bucket = sorted_idxs[i : i + self.batch_size]
            if len(bucket) == self.batch_size or not self.drop_last:
                buckets.append(bucket)

        np.random.shuffle(buckets)

        for bucket in buckets:
            yield list(bucket)

    def __len__(self):
        if self.drop_last:
            return len(self.lengths) // self.batch_size
        else:
            return (len(self.lengths) + self.batch_size - 1) // self.batch_size


def bucket_dataloader(dataset, batch_size: int = 128) -> DataLoader:
    lengths = [min(x.shape[0], dataset.max_frames) for x in dataset.video_coordinates]

    sampler = BucketBatchSampler(lengths, batch_size=batch_size, drop_last=False)

    return DataLoader(dataset, batch_sampler=sampler, collate_fn=collate_fn)

In [10]:
def get_data_loaders() -> Tuple[DataLoader, DataLoader]:
    """
    Reads dataloaders from the normalized coordinates

    Returns:
        Tuple[DataLoader, DataLoader]: train and test dataloaders
    """
    train_df = pd.read_csv(TRAIN_FILE)

    train_df["sign"] = train_df["sign"].map(SIGN2INDEX_JSON)
    train_split, test_split = train_test_split(
        train_df, test_size=0.1, stratify=train_df["sign"], random_state=42
    )

    all_train_videos = []
    for path in train_split["path"].to_list():
        all_train_videos.append(frame_stacked_data(path))

    all_test_videos = []
    for path in test_split["path"].to_list():
        all_test_videos.append(frame_stacked_data(path))

    np.savez_compressed(
        "asl_preprocessed.npz",
        train_videos=np.array(all_train_videos, dtype=object),
        train_labels=train_split["sign"].to_numpy(),
        test_videos=np.array(all_test_videos, dtype=object),
        test_labels=test_split["sign"].to_numpy(),
    )

    train_dataset = ASLDataset(all_train_videos, train_split["sign"].to_numpy(), augment=True)
    test_dataset = ASLDataset(all_test_videos, test_split["sign"].to_numpy())

    train_loader = bucket_dataloader(train_dataset)
    test_loader = bucket_dataloader(test_dataset)

    return train_loader, test_loader

In [11]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[: x.size(1)]

In [12]:
class VideoTransformer(nn.Module):
    def __init__(
        self, input_dim, num_classes, d_model=192, n_heads=4, n_layers=4, dropout=0.1
    ):
        super().__init__()

        self.pos_enc = PositionalEncoding(d_model)
        self.input_proj = nn.Linear(input_dim, d_model)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
        )

        self.encoder = nn.TransformerEncoder(encoder_layer, n_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)

    def forward(self, x, mask):
        B, T, _ = x.shape

        x = self.input_proj(x)
        x = self.pos_enc(x)

        cls = self.cls_token.expand(B, 1, -1)
        x = torch.cat([cls, x], dim=1)

        cls_mask = torch.ones(B, 1, device=mask.device, dtype=torch.bool)
        mask = torch.cat([cls_mask, mask], dim=1)

        x = self.encoder(x, src_key_padding_mask=~mask)
        x = self.norm(x[:, 0])

        return self.head(x)

In [13]:
def train_epoch(data_loader: DataLoader) -> float:
    """
    Trains the model with the train dataloader

    Args:
        data_loader (DataLoader): DataLoader with the train frames

    Returns:
        float: Train loss for the epoch
    """
    model.train()
    train_loss = 0

    for x, mask, y in data_loader:
        x, mask, y = x.to(device), mask.to(device), y.to(device)

        optimizer.zero_grad()
        with autocast(enabled=use_amp, device_type=device.type):
            logits = model(x, mask)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

    return train_loss / len(data_loader)


@torch.no_grad()
def test_epoch(data_loader: DataLoader) -> Tuple[float, float]:
    """
    Tests the model with the test dataloader

    Args:
        data_loader (DataLoader): DataLoader with the test frames

    Returns:
        float: Test loss for the epoch
    """
    model.eval()
    test_loss = 0
    correct, total = 0, 0

    for x, mask, y in data_loader:
        x, mask, y = x.to(device), mask.to(device), y.to(device)

        logits = model(x, mask)

        loss = criterion(logits, y)
        test_loss += loss.item()

        prediction = logits.argmax(dim=1)
        correct += (prediction == y).sum().item()
        total += y.size(0)

    return test_loss / len(data_loader), correct / total

In [14]:
model = VideoTransformer(
    input_dim=2 * len(ALL_COLUMNS), num_classes=len(SIGN2INDEX_JSON)
).to(device)

optimizer = optim.AdamW(model.parameters(), lr=3e-4)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler = GradScaler(enabled=use_amp)

scheduler = ReduceLROnPlateau(optimizer, factor=0.5, patience=3, min_lr=1e-6)

train_loader, test_loader = get_data_loaders()

epochs = 100
least_test_loss = float("inf")
patience = 0

In [15]:
for epoch in range(epochs):
    train_loss = train_epoch(train_loader)
    test_loss, test_accuracy = test_epoch(test_loader)

    scheduler.step(test_loss)

    print(
        f"Epoch: {epoch} | "
        f"Train Loss: {train_loss} | "
        f"Test Loss: {test_loss} | "
        f"Test Accuracy: {test_accuracy}"
    )

    if test_loss < least_test_loss:
        least_test_loss = test_loss
        patience = 0
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "test_loss": test_loss,
            },
            "best_model.pth",
        )
    else:
        patience += 1

    if patience > 15:
        break

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/transformer.py:508: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Epoch: 0 | Train Loss: 5.137370054345382 | Test Loss: 4.735693158330144 | Test Accuracy: 0.056731583403895


Epoch: 1 | Train Loss: 4.3570269832037445 | Test Loss: 3.9179560944840714 | Test Accuracy: 0.21390770533446232


Epoch: 2 | Train Loss: 3.7288820230871216 | Test Loss: 3.48270050255028 | Test Accuracy: 0.3013336155800169


Epoch: 3 | Train Loss: 3.320917376898285 | Test Loss: 3.2154348863137736 | Test Accuracy: 0.3627222692633362


...

Epoch: 49 | Train Loss: 1.2709200756890433 | Test Loss: 1.8315925308175989 | Test Accuracy: 0.7599491955969517


Epoch: 50 | Train Loss: 1.2629818962928945 | Test Loss: 1.83522494419201 | Test Accuracy: 0.7578323454699407


Epoch: 51 | Train Loss: 1.2590146471683243 | Test Loss: 1.8424601071589701 | Test Accuracy: 0.7588907705334462


Epoch: 52 | Train Loss: 1.2509981694974397 | Test Loss: 1.8232981192099083 | Test Accuracy: 0.7613251481795089


Epoch: 53 | Train Loss: 1.1750078545477158 | Test Loss: 1.7645573632137195 | Test Accuracy: 0.7776248941574937
